In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    to_date,
    to_timestamp,
    when,
    current_timestamp,
    from_json
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    IntegerType
)


# ============================================
# STORAGE PATHS
# ============================================

bronze_root = (
    "abfss://bronze@bankingdelakevishal.dfs.core.windows.net/"
)

silver_root = (
    "abfss://silver@bankingdelakevishal.dfs.core.windows.net/"
)

In [0]:
# ============================================
# CONFIGURATION & SCHEMA SETUP
# ============================================
CATALOG_NAME = "banking_lakehouse_db2"
SCHEMA_NAME = "silver"
TABLE_NAME = "branch"
FULL_TABLE_NAME = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{TABLE_NAME}"

# Ensure Silver schema exists inside your Unity Catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

# ============================================
# READ BRONZE DATA
# ============================================
branch_bronze_df = spark.read.parquet(f"{bronze_root}branch/")

# ============================================
# TRANSFORM & CLEANSE (BRONZE → SILVER)
# ============================================
branch_silver_df = (
    branch_bronze_df
    .select(
        trim(col("branch_id")).alias("branch_id"),
        trim(col("branch_name")).alias("branch_name"),
        upper(trim(col("ifsc_code"))).alias("ifsc_code"),
        trim(col("city")).alias("city"),
        trim(col("state")).alias("state"),
        upper(trim(col("region"))).alias("region"),
        to_date(trim(col("opening_date")), "yyyy-MM-dd").alias("opening_date")
    )
    .filter(col("branch_id").isNotNull())
    .dropDuplicates(["branch_id"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)

# ============================================
# WRITE FRESH DATA (OVERWRITE PATH & TABLE)
# ============================================
# 1. Overwrite raw Delta files in ADLS
(
    branch_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{silver_root}branch/")
)

# 2. Overwrite / Register managed Unity Catalog table
(
    branch_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)

# ============================================
# VERIFICATION METRICS
# ============================================
record_count = branch_silver_df.count()
print(f"Branch Silver Count: {record_count} records saved to '{FULL_TABLE_NAME}'.")

Branch Silver Count: 500 records saved to '`banking_lakehouse_db2`.silver.branch'.
